# 00 · 从可验证的环境开始

目标：确认当前内核就是 CNN-Tutorial，能够使用 PyTorch 与 GPU，并找到项目目录。
依次运行每个 cell；本章不下载数据、不训练。第一次使用请先阅读 `docs/01_setup/environment.md`。

## 1. 项目根目录

Jupyter 的工作目录可能是项目根目录，也可能是 notebooks。向上查找 MAIN.md 和源码目录，比硬编码个人电脑路径更方便复用。

In [1]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "cnn_tutorial").is_dir() and (p / "MAIN.md").exists()), None)
if ROOT is None:
    raise RuntimeError("请从 CNN-Tutorial 项目目录打开此 notebook")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Project:", ROOT.name)

Project: CNN-Tutorial


## 2. 核实当前解释器与依赖

`sys.executable` 决定真正运行代码的环境；电脑装有 CUDA 并不自动意味着当前 torch 能使用 GPU。
`torch.version.cuda` 是这个 PyTorch 构建使用的 CUDA runtime 版本。

In [2]:
import platform
import importlib.metadata as metadata
import torch
packages = ["torch", "numpy", "Pillow", "notebook", "ipykernel", "matplotlib", "nbformat", "nbclient"]
environment = {
    "python": platform.python_version(),
    "packages": {name: metadata.version(name) for name in packages},
    "cuda_runtime": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
print("Interpreter environment:", Path(sys.executable).parent.name)
environment

Interpreter environment: CNN-Tutorial


{'python': '3.14.8',
 'packages': {'torch': '2.14.1+cu132',
  'numpy': '2.5.2',
  'Pillow': '12.3.0',
  'notebook': '7.6.3',
  'ipykernel': '7.4.0',
  'matplotlib': '3.11.2',
  'nbformat': '5.11.1',
  'nbclient': '0.11.0'},
 'cuda_runtime': '13.2',
 'cuda_available': True,
 'gpu': 'NVIDIA GeForce RTX 5060 Laptop GPU'}

## 3. 真正执行一次设备计算

创建张量并在设备上进行矩阵乘法，比只打印 GPU 名称多验证一步。CPU 也能学习本教程，只是训练速度可能较慢。

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x = torch.arange(12, dtype=torch.float32, device=device).reshape(3, 4)
result = x @ x.T
assert result.shape == (3, 3)
print("Device:", device)
print(result.cpu())

Device: cuda
tensor([[ 14.,  38.,  62.],
        [ 38., 126., 214.],
        [ 62., 214., 366.]])


## 4. 保存环境快照

实验结果需要知道使用了哪个框架版本。快照只保存必要版本信息，不保存账号、令牌或整个环境变量。

In [4]:
import json
snapshot_dir = ROOT / "artifacts" / "environment"
snapshot_dir.mkdir(parents=True, exist_ok=True)
(snapshot_dir / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
print("Saved:", (snapshot_dir / "environment.json").relative_to(ROOT))

Saved: artifacts\environment\environment.json


## 小结与下一步

现在打开 `01_gray_gesture_cnn.ipynb`，逐层观察图像如何变成类别分数。

### 语法小词典

- `Path / "name"`：拼接路径，避免手写反斜杠。
- `dict`：键值记录，适合保存环境与实验配置。
- `assert`：不满足约定时立即报错，帮助尽早发现输入问题。
- `x @ x.T`：矩阵乘法；这里的 `.T` 只用于二维张量。